In [1]:

import pandas as pd

import pathlib
import numpy as np


In [ ]:
DB_DIR = pathlib.Path("../database/dbs")
DB_DIR.mkdir(parents=True, exist_ok=True)

VW_MODEL_DB_PATH = DB_DIR / "vw_model_db.csv"
VW_MODEL_DB_PATH = VW_MODEL_DB_PATH.resolve()


model_db_df = pd.read_csv(VW_MODEL_DB_PATH)




# JSON template for type mapping
column_type_template = {
    "string": ["Description", "Description2", "model"],
    "int": ["Year"],
    "float": ["MSRP"],
    # "boolean": ["is_active_model"],
    "datetime": ["created_at", "updated_at"]
}


# Essential columns
vw_essential_columns = ['Model Year', 'model', 'Part #',
       'English Description', 'French description', 'MSRP', 'Labour in hours', "trim_level","comments_en", "comments_fr"]

vw_to_standard_columns_names = {
    "model": "model_name",
    'Model Year': 'model_year',
    'Part #': 'part_number',
    'English Description': 'part_description_en',
    'French description': 'part_description_fr',
    'Labour in hours': 'install_hours',
    'MSRP':'MSRP',
    "Labour in hours": 'labour_hours',
    "trim_level": "trim_level",
    "comments_en": "comments_en",
    "comments_fr": "comments_fr"
}


# Finally convert columns to meet Rate Importer

COLUMN_MAPPING_FOR_RATE_IMPORTER = {
    "model_year": "Year",
    "short_model_number": "Model",
    "trim_level": "Package",
    "part_number": "Part",
    "part_description_en": "Description",
    "comments_en": "Comments",
    'part_description_fr': 'Description',
    'comments_fr': 'Comments',
    'MSRP': 'Price',
    'install_hours': 'Hours'
}
 

In [ ]:
def apply_type_mapping(df, type_mapping):
    """
    Convert columns in df using a config structure like:
    {"datatype": ["col1", "col2"]}
    Supported datatypes: string, int, float, boolean, datetime.
    """

    df = df.copy()
    for dtype, cols in type_mapping.items():
        for col in cols:
            if col not in df.columns:
                continue
            if dtype in {"string", "str", "object"}:
                df[col] = df[col].astype("string")
            elif dtype in {"int", "int64"}:
                df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
            elif dtype in {"float", "float64", "double"}:
                df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
            elif dtype in {"boolean", "bool"}:
                df[col] = df[col].astype("boolean")
            elif dtype in {"datetime", "datetime64", "datetime64[ns]"}:
                df[col] = pd.to_datetime(df[col], errors="coerce")
            else:
                df[col] = df[col].astype(dtype)
    return df


In [167]:
def standardize_column_names(df, column_mapping):
    """
    Standardize column names in df using a mapping dict {current_name: new_name}.
    Prints the changes made and returns the updated DataFrame.
    Handles missing columns by warning and skipping.
    """
    df = df.copy()
    original_columns = df.columns.tolist()
    renamed_columns = []
    missing_columns = []

    for current_name, new_name in column_mapping.items():
        if current_name in df.columns:
            df.rename(columns={current_name: new_name}, inplace=True)
            renamed_columns.append(f"'{current_name}' -> '{new_name}'")
        else:
            missing_columns.append(current_name)

    if renamed_columns:
        print("Column renames applied:")
        for change in renamed_columns:
            print(f"  {change}")
    else:
        print("No columns were renamed.")

    if missing_columns:
        print("Warning: The following columns in the mapping were not found in the DataFrame:")
        for col in missing_columns:
            print(f"  '{col}'")

    return df

In [69]:


def capitalize_text_columns(df, columns):
    """
    Return a copy of df with the specified columns converted to uppercase text.
    Columns that are missing are skipped. Non-text values are converted safely.
    """
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = df[col].astype("string").str.upper()
    return df


In [ ]:

def load_model_db(model_db_path):
    """
    Load the model database from the specified path and return a DataFrame.
    The model database is expected to have columns: 'model' and 'trim_level'.
    """
    model_db_df = pd.read_csv(model_db_path)
    return model_db_df

def load_data(vw_file_path):
    """
    Load the VW data from the specified Excel file and return a DataFrame.
    The data is expected to be in the sheet named '2026_Acc_Master_Checked FINAL'.
    """
    vw_df = pd.read_excel(vw_file_path, sheet_name="2026_Acc_Master_Checked FINAL")
    return vw_df



In [73]:
# Strip whitespace from column names and values in the combined dataframe
def strip_whitespace(df):
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].str.strip()
    return df

In [133]:

def slice_for_essential_columns(df, essential_columns):
    missing_columns = [col for col in essential_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing essential column: {missing_columns}")
    return df[essential_columns].copy()

In [182]:
# def reduce_to_essential_columns(df, essential_columns):
#     missing_columns = [col for col in essential_columns if col not in df.columns]
#     if missing_columns:
#         raise ValueError(f"Missing essential column: {missing_columns}")
#     return df[essential_columns].copy()

In [183]:
def clean_data(df, skip_slicing = False):
    
    if not skip_slicing:
        df = slice_for_essential_columns(df, essential_columns=vw_essential_columns)
    df = strip_whitespace(df)
    df = apply_type_mapping(df, column_type_template)
    df = capitalize_text_columns(df, ["model", "Description", "Description2"])

    return df

In [75]:
def get_trim_level_list(model_name):

    return model_db_df[model_db_df['Description'].str.contains(model_name, case=False, na=False)]['Description2'].tolist()

In [127]:
def populate_dummy_columns(vw_df):
    vw_df["MSRP"]= pd.NA
    vw_df["MSRP"] = vw_df["MSRP"].apply(lambda x: np.random.randint(50, 501) if pd.isnull(x) else x)
    # Create 2 new columns named "comments_eng" and comments_fr and fill them with with the text in the "English Description" and "French description" columns respectively. 
    vw_df["comments_en"] = vw_df["English Description"]
    vw_df["comments_fr"] = vw_df["French description"]
    return vw_df


def populate_dummy_trim_levels(vw_df, model_db_df):
    unique_model_names_in_db = model_db_df['Description'].unique()
    missing_model_names = []

    for (model, year), group in vw_df.groupby(['model', 'Model Year']):
        is_model_name_in_db = any(str(model).lower() in str(db_model).lower() for db_model in unique_model_names_in_db)

        if is_model_name_in_db:
            sampled_trim_levels = get_trim_level_list(model)
            vw_df.loc[group.index, 'trim_level'] = np.random.choice(sampled_trim_levels, size=len(group), replace=True)
        else:
            missing_model_names.append(model)
    print("Missing model names in model_db_df:", missing_model_names)

    vw_df['trim_level'] = vw_df['trim_level'].str.upper()
    return vw_df



In [141]:
def add_dummy_data(vw_df, model_db_df):
    vw_df = populate_dummy_columns(vw_df)
    vw_df = populate_dummy_trim_levels(vw_df, model_db_df)

    return vw_df

In [ ]:

# def save_to_excel(data_dict, file_path):
#     with pd.ExcelWriter(file_path) as writer:
#         for year, models in data_dict.items():
#             for model, trims in models.items():
#                 english_df = pd.DataFrame()
#                 french_df = pd.DataFrame()
#                 for trim, languages in trims.items():
#                     english_df = pd.concat([english_df, languages["en"]], ignore_index=True)
#                     french_df = pd.concat([french_df, languages["fr"]], ignore_index=True)
#                 english_sheet_name = f"{year}_{model}_en"
#                 french_sheet_name = f"{year}_{model}_fr"
#                 english_df.to_excel(writer, sheet_name=english_sheet_name, index=False)
#                 french_df.to_excel(writer, sheet_name=french_sheet_name, index=False)
                

In [ ]:
# def save_to_excel(data_dict, file_path):
#     with pd.ExcelWriter(file_path) as writer:
#         for year, models in data_dict.items():
#             for model, trims in models.items():
#                 for trim, languages in trims.items():
#                     sheet_name_en = f"{year}_{model}_{trim}_en"
#                     sheet_name_fr = f"{year}_{model}_{trim}_fr"
#                     languages["en"].to_excel(writer, sheet_name=sheet_name_en, index=False)
#                     languages["fr"].to_excel(writer, sheet_name=sheet_name_fr, index=False)



In [ ]:

# # the name of the file will need to contain the time it was created, to make it easier to manage the files and to keep track of the different versions of the feed. We can use the datetime module in Python to get the current date and time, and then we can format it to include in the file name. This will allow us to have a clear and organized way to manage the different versions of the feed, and it will also make it easier to identify the most recent version of the feed when we need to use it for the integration with the dealer portal.

# time = pd.Timestamp.now().strftime("%Y%m%d-%H%M%S")

# # first check if the directory exists, if not create it
# output_directory = pathlib.Path("../ready_to_upload/2026/Mazda")
# output_directory.mkdir(parents=True, exist_ok=True)

# mazda_ready_to_load_excel_file = output_directory / f"mazda_accy_ready_to_upload_{time}.xlsx"

# save_to_excel(mazda_model_trim_feed, mazda_ready_to_load_excel_file)

In [102]:
'''
Items to test for data integrity:
    1. Check if the number of rows in the loaded feed matches the number of rows in the original dataframe for a specific model and trim level. This will help us to ensure that all the data is correctly saved and loaded, and that there are no missing or duplicated rows in the feed.    
    2. Check if the values in specific columns, such as part numbers, descriptions, and prices, match between the original dataframe and the loaded feed for a sample of rows. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process.
    3. Check if the data types of the columns in the loaded feed match the data types of the columns in the original dataframe. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process.
    4. Check if there are any null values in the loaded feed for columns that are not supposed to have null values, such as part numbers and descriptions. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process.
    5. Check if the values in the loaded feed are correctly formatted, such as checking if the part numbers have the correct format and if the prices are in the correct format. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process.
    6. Check if the values in the loaded feed are correctly mapped to the correct columns, such as checking if the part numbers are in the part number column and if the descriptions are in the description column. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process.
    7. Check if the values in the loaded feed are correctly grouped by model and trim level, such as checking if the part numbers for a specific model and trim level are all in the same sheet in the excel file. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process.
    8. Check if the values in the loaded feed are correctly translated, such as checking if the descriptions in English match the descriptions in French for the same part number. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process.
    9. Check if the values in the loaded feed are correctly calculated, such as checking if the install hours are correctly calculated based on the original dataframe. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process.
    10. Check if the values in the loaded feed are correctly sorted, such as checking if the part numbers are sorted in the same order as in the original dataframe. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process.
    11. Check if the values in the loaded feed are correctly filtered, such as checking if only the rows that meet the package format check are included in the feed. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process.
    12. Check if the values in the loaded feed are correctly aggregated, such as checking if the part numbers for a specific model and trim level are correctly aggregated in the same sheet in the excel file. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process.
    13. Check if the values in the loaded feed are correctly formatted for the integration with the dealer portal, such as checking if the column names and data types match the requirements for the dealer portal. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process, which could affect the integration with the dealer portal.
    14. Check if the values in the loaded feed are correctly versioned, such as checking if the file name contains the correct time stamp and if the most recent version of the feed is being used for the integration with the dealer portal. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process, which could affect the integration with the dealer portal.

'''


'\nItems to test for data integrity:\n    1. Check if the number of rows in the loaded feed matches the number of rows in the original dataframe for a specific model and trim level. This will help us to ensure that all the data is correctly saved and loaded, and that there are no missing or duplicated rows in the feed.    \n    2. Check if the values in specific columns, such as part numbers, descriptions, and prices, match between the original dataframe and the loaded feed for a sample of rows. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process.\n    3. Check if the data types of the columns in the loaded feed match the data types of the columns in the original dataframe. This will help us to ensure that the data is correctly saved and loaded, and that there are no issues with the data integrity during the save and load process.\n    4. Check if there are any null values in the 

In [ ]:
# # Test function

# def test_data_integrity(original_df, loaded_data, sample_size=10):
#     original = original_df.rename(columns={
#         'ModelYear': 'model_year',
#         'CarLineCode': 'model',
#         'TrimLevel': 'model_number',
#         'PartNumber': 'part_number',
#         'AccessoryName': 'part_description_en',
#         'AccessoryNameFR': 'part_description_fr',
#         'RestrictionEn': 'comments_en',
#         'RestrictionFr': 'comments_fr',
#         'StandardInstallTime': 'install_hours'
#     }).copy()

#     if 'package' not in original.columns and 'model_number' in original.columns:
#         original['package'] = original['model_number'].astype(str).str[-4:]
#     if 'short_model_number' not in original.columns and 'model_number' in original.columns:
#         original['short_model_number'] = original['model_number'].astype(str).str[:-4]

#     for col in ['comments_en', 'comments_fr', 'part_description_en', 'part_description_fr']:
#         if col in original.columns:
#             original[col] = original[col].fillna('').astype(str)

#     if isinstance(loaded_data, (str, pathlib.Path)):
#         sheets = pd.read_excel(loaded_data, sheet_name=None)
#     elif isinstance(loaded_data, dict):
#         sheets = loaded_data
#     elif isinstance(loaded_data, pd.DataFrame):
#         sheets = {'loaded_sheet': loaded_data}
#     else:
#         raise TypeError("loaded_data must be a file path, a dict of sheets, or a DataFrame")

#     for sheet_name, sheet_df in sheets.items():
#         if sheet_df.empty:
#             continue

#         sheet_df = sheet_df.copy()
#         sheet_df.columns = [c.lower() for c in sheet_df.columns]

#         for col in ['comments_en', 'comments_fr', 'part_description_en', 'part_description_fr']:
#             if col in sheet_df.columns:
#                 sheet_df[col] = sheet_df[col].fillna('').astype(str)

#         if 'package' not in sheet_df.columns and 'model_number' in sheet_df.columns:
#             sheet_df['package'] = sheet_df['model_number'].astype(str).str[-4:]
#         if 'short_model_number' not in sheet_df.columns and 'model_number' in sheet_df.columns:
#             sheet_df['short_model_number'] = sheet_df['model_number'].astype(str).str[:-4]

#         lang = None
#         if sheet_name.lower().endswith('_en'):
#             lang = 'en'
#         elif sheet_name.lower().endswith('_fr'):
#             lang = 'fr'

#         parts = sheet_name.split('_')
#         year = None
#         model = None
#         if len(parts) >= 3:
#             try:
#                 year = int(parts[0])
#                 model = parts[1]
#             except ValueError:
#                 pass

#         subset = original
#         if year is not None:
#             subset = subset[subset['model_year'] == year]
#         if model is not None:
#             subset = subset[subset['model'] == model]

#         if subset.empty:
#             raise AssertionError(f"No matching original rows found for sheet {sheet_name}")

#         sample_n = min(sample_size, len(subset))
#         sample = subset.sample(n=sample_n, random_state=42)

#         for _, row in sample.iterrows():
#             condition = (
#                 (sheet_df['part_number'] == row['part_number']) &
#                 (sheet_df['package'] == row['package']) &
#                 (sheet_df['short_model_number'] == row['short_model_number'])
#             )
#             loaded_row = sheet_df[condition]
#             assert not loaded_row.empty, (
#                 f"Part number {row['part_number']} not found in sheet {sheet_name}"
#             )
#             loaded_row = loaded_row.iloc[0]

#             if lang == 'en':
#                 assert loaded_row['part_description_en'] == row['part_description_en'], (
#                     f"{sheet_name}: English description mismatch for part {row['part_number']}"
#                 )
#                 assert loaded_row['comments_en'] == row.get('comments_en', ''), (
#                     f"{sheet_name}: English comments mismatch for part {row['part_number']}"
#                 )
#             elif lang == 'fr':
#                 assert loaded_row['part_description_fr'] == row['part_description_fr'], (
#                     f"{sheet_name}: French description mismatch for part {row['part_number']}"
#                 )
#                 assert loaded_row['comments_fr'] == row.get('comments_fr', ''), (
#                     f"{sheet_name}: French comments mismatch for part {row['part_number']}"
#                 )

#             assert float(loaded_row['msrp']) == float(row['MSRP']), (
#                 f"{sheet_name}: MSRP mismatch for part {row['part_number']}"
#             )
#             if 'install_hours' in sheet_df.columns:
#                 assert float(loaded_row['install_hours']) == float(row['install_hours']), (
#                     f"{sheet_name}: install_hours mismatch for part {row['part_number']}"
#                 )

#     return True




,Model,Description,Description2,is_active_model
0,BU52RS,Jetta,Trendline,True
1,BU53RS,Jetta,Comfortline,True
2,BU54RS,Jetta,Highline,True
3,BU59V2,Jetta GLI,GLI,True
4,CA33PR,Atlas,Comfortline,True


In [190]:
# Loaded VW data
vw_file_path = "../landing_zone/2026/VW/VW - MY26 Accessories_Master File -2.23.2026.xlsx"
vw_data_df = load_data(vw_file_path)

In [ ]:

model_db_df = load_model_db(VW_MODEL_DB_PATH)
  

In [191]:

vw_df_cleaned = clean_data(vw_data_df, skip_slicing=True) # skip_slicing param will only be added for testing purposes, because we are augmenting data to the feed. Once the feed comes with everything we need, we will remove it, and the script will slice the DF by default, to only focus on.

vw_df_augmented = add_dummy_data(vw_df_cleaned, model_db_df)
vw_df_ready_to_use = slice_for_essential_columns(vw_df_augmented, essential_columns=vw_essential_columns)
vw_df_ready_to_use = standardize_column_names(vw_df_ready_to_use, vw_to_standard_columns_names)

 

Missing model names in model_db_df: ['TIGUAN-NF']


In [198]:
model_db_df.head()

,Model,Description,Description2,is_active_model
0,BU52RS,Jetta,Trendline,True
1,BU53RS,Jetta,Comfortline,True
2,BU54RS,Jetta,Highline,True
3,BU59V2,Jetta GLI,GLI,True
4,CA33PR,Atlas,Comfortline,True


In [199]:
vw_df_ready_to_use.head()

,model_year,model_name,part_number,part_description_en,part_description_fr,MSRP,labour_hours,trim_level,comments_en,comments_fr
0,2026,JETTA,5GM071496 8Z8,"16"" Merano Wheel – silver","Roue Merano de 16"" – Argent",197,1.6,COMFORTLINE,"16"" Merano Wheel – silver","Roue Merano de 16"" – Argent"
1,2026,JETTA,5GM601025E FZZ,"16"" Rama Wheel","Roues Rama de 16""",70,1.6,GLI,"16"" Rama Wheel","Roues Rama de 16"""
2,2026,TAOS,2GA071497A DM9,"17"" Gavia, Adamantium Dark Metallic","Roue Gavia de 17"", Adamantium foncé métallisé",224,1.6,TRENDLINE,"17"" Gavia, Adamantium Dark Metallic","Roue Gavia de 17"", Adamantium foncé métallisé"
3,2026,GTI,5H0071497 8Z8,"17"" GTI Winter Wheel","Roues de 17"" GTI pour l’hiver",331,1.6,GTI,"17"" GTI Winter Wheel","Roues de 17"" GTI pour l’hiver"
4,2026,TAOS,2GA071497 DM9,"17"" Merano Winter Wheel - Adamantium Dark Meta...","Roue Merano de 17"", Adamantium métallique foncé",486,1.6,COMFORTLINE,"17"" Merano Winter Wheel - Adamantium Dark Meta...","Roue Merano de 17"", Adamantium métallique foncé"


In [ ]:
def fetch_model_number_by_modelName_and_year(model_name, year, model_db_df):
    """
    Fetch the model number from the model database based on the model name and year.
    Returns the model number if found, otherwise returns None.
    """
    matching_rows = model_db_df[
        (model_db_df['Description'].str.contains(model_name, case=False, na=False)) &
        (model_db_df['Year'] == year)
    ]
    if not matching_rows.empty:
        return matching_rows.iloc[0]['model']
    return None

In [ ]:
def generate_ready_to_upload_dfs(vw_df):

    # Creating a model and trim specific feed for Mazda, which will be used for the integration with the dealer portal. This will create a dict that will contains {year: model: trim: dataframe} structure, which will be used to create the feed for each model and trim.

    VW_model_trim_feed = {} 

    for year in vw_df['model_year'].unique():
        year_df = vw_df[vw_df['model_year'] == year]
        VW_model_trim_feed[year] = {}
        for model in year_df['model'].unique():
            model_df = year_df[year_df['model'] == model]
            VW_model_trim_feed[year][model] = {}
            for trim in vw_df['trim_level'].unique():
                trim_df = model_df[model_df['trim_level'] == trim]
                english_df = trim_df[["model_year", "model", "trim_level", 'part_number', 'part_description_en', 'comments_en', 'MSRP', 'install_hours']].reset_index(drop=True) 
                french_df = trim_df[["model_year", "model", "trim_level", 'part_number', 'part_description_fr', 'comments_fr', 'MSRP', 'install_hours']].reset_index(drop=True)
                english_df.index = english_df.index + 1
                french_df.index = french_df.index + 1

                VW_model_trim_feed[year][model][trim] = {
                    "en": english_df,
                    "fr": french_df
                }

    return VW_model_trim_feed
